# K4TQ-D1 "Dorothy" — folding the wings in flight to pass a smoke zone

**Dorothy** is Freddy (K4TQ-F1: front NACA 4415, rear NACA 2415) with a **powered fold**: at each hinge a self-locking worm
gearmotor and a 6:1 sector (~15 N·m), a fail-safe solenoid lock in place of the manual pin. **K4TQ-D2** adds the Jetson.

The question: can she pass a **smoke zone** (a fire's plume — hot, turbulent, little to see) with the wings folded, and
**keep her manoeuvrability**? Two folds, both keeping the propellers working:

- **cruise V-tuck** (tilt 0): the wings fold *up* about their hinge pins; the thrust stays along the chord;
- **narrow hover** (tilt 90): the wings swing *aft*; the discs stay horizontal (quad-like control, smaller footprint).

Labels: ESTIMATE / CALC (MOCK where the BEMT drive maps are coarsened under `DEBUG = True`). Short study, a minute.

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
import k4tq, k4tq_sizing as sz, k4tq_compare as kc, k4tq_cad
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
D1 = sz.vehicle("D1")
print(k4tq.VARIANTS["D1"]["title"]); print(f"{D1['mass']['mass_kg']:.2f} kg, masses {D1['p']['cad_mass_status']}")

## 1. What the fold drive costs — Dorothy against Freddy

In [ ]:
ev = {v: kc.evaluate(v, corridor=False) for v in ("F1", "F2", "D1", "D2")}
kc.table(ev).T

## 2. The smoke zone: 150 m of air 50 K hotter, 6 m/s gusts (cruise V-tuck)

In [ ]:
sm = sz.smoke_transit(D1)
sm.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ok = sm[sm["within the propellers' speed"]]
for col in ("gust bump (× unfolded)", "roll authority (× unfolded)", "pitch authority (× unfolded)", "yaw authority (× unfolded)"):
    ax[0].plot(sm.index, sm[col], "o-", label=col)
ax[0].axvspan(ok.index.max(), sm.index.max(), color="0.9", label="beyond the propellers' speed")
ax[0].set(xlabel="fold [deg]", title="what the tuck buys (gusts) and what it costs (control)"); ax[0].legend(fontsize=8)
Vc = sm.attrs["V_common"]
ax[1].plot(sm.index, sm[f"load factor at {Vc:.0f} m/s"], "o-", label=f"load factor at {Vc:.0f} m/s")
ax[1].plot(sm.index, sm["transit speed [m/s]"] / 10, "s--", label="transit speed / 10 [m/s]")
ax[1].set(xlabel="fold [deg]", title="manoeuvring margin at one speed"); ax[1].legend(fontsize=8)

## 3. The narrow hover (tilt 90): footprint against control arms

In [ ]:
sz.hover_fold(D1).round(3)

## 4. Does folding make drag? Can it go faster? (level flight, sea level, four motors)

In [ ]:
fl = sz.fold_level(D1)
display(fl["electrical [W]"].unstack(0).round(0))
fl["drag [N]"].unstack(0).round(2)

## 5. The fold drive's load in the tuck

In [ ]:
sz.fold_drive_loads(D1)

## 6. The two folds in CAD

In [ ]:
png = k4tq_cad.render_dorothy(kc.DATA / "figures" / "k4tq_D1_folds.png", D1["p"], items=D1["items"])
Image(str(png))

## Reading it

- **Smoke transit: yes, with a partial tuck of 20–30°.** At 30° the gust bump falls to 75 % of the unfolded wing's,
  roll authority keeps ~65 %, pitch 75 %, yaw 87 %; 150 m take ~5.5 s. Beyond ~35° the folded wings' stall speed in the
  hot air rises above what the 16x10 propellers can still push (~28 m/s): 45° and 60° are not flyable on the wings.
- **The narrow hover is the manoeuvrable way through a tight gap**: folded 30° at tilt 90 the width over the discs
  drops 1.65 → 1.50 m and the control is still a quadcopter's (roll arms ×0.88, pitch and yaw unchanged) — slow,
  so longer in the smoke and on hover power.
- **Folding makes drag, it does not remove it**: the wetted area (profile drag) stays, and the same lift on a shorter
  projected span costs more induced drag (×1/cos² fold). It cannot go faster; the top speed is the propellers' (~28 m/s).
- **The price**: +0.66 kg; Dorothy's sagged-pack T/W falls below 1.6 on 16x10 (17x10 restores it, at a thinner
  mission margin); D2 (with the Jetson) no longer closes the standard 1 kg mission.